# Caution

This is an important historical fast workflow, but some cells use the old fast triple-barrier construction. The old barrier was not side-symmetric, secondary gross-exposure normalisation did not necessarily preserve dollar neutrality, and the DGP volatility parameter targets equal-weight portfolio volatility rather than individual stock volatility. Use current conclusions from the side-symmetric and fundamental diagnostic notebooks.


## Explanation


## Data generation

In [499]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


rng = np.random.default_rng(0)


# Parameter specification with scales chosen to be realistic for daily returns

N = 500
T = 10000
K = 10

dates = pd.date_range("1986-01-01", periods=T, freq="B")
cols = [f"Stock_{i:03d}" for i in range(N)]

target_daily_vol = 0.10/np.sqrt(252)  # target annualized volatility (converted to daily)
# split total variance (vol**2) in factor and specific components
split = [0.6, 0.4]  # 60% factor, 40% specific

trend_daily = (0.50/252) # trend return 

print(f"Target total daily volatility: {target_daily_vol:.4f}"  )
print(f"Target total annual volatility: {target_daily_vol*np.sqrt(252):.4f}"  )

n_up, n_down = 200, 100       # number of stocks with an upward/downward trend



Target total daily volatility: 0.0063
Target total annual volatility: 0.1000


In [500]:
wi = np.ones([N,1])/N  # define equal weights for all stocks in the universe
# build covariaance matrix and beta and generate random process of returns

Z = np.random.normal(0, 1, size=(N, K-1))    # factor loadings with mean 0 and std 1
Z = ((Z - Z.mean(axis=1, keepdims=True)) / Z.std(axis=1, keepdims=True)) # standardize factor loadings
Z = np.concatenate([np.random.uniform(0.5, 1.5, size=(N, 1)), Z], axis=1)  # market factor with positive loading of around 1 for all stocks

G = np.diag(np.concatenate([np.ones(1), [0.25 * (np.exp(-i)) for i in range(K-1)]]))  # factor covariance matrix with decaying eigenvalues to be re-scaled to match the target factor variance
# G = np.diag(np.concatenate([np.ones(1),np.linspace(0.16,0.01,num=K-1)]))
S = np.random.uniform(0.01, 0.25, size=N) # specific variances to be re-scaled to match the target specific variance

# Rescaling factor and computing the covariance matrix
multf = np.squeeze((split[0]) * target_daily_vol**2 / (wi.T @ (Z @ G @ Z.T) @ wi))
mults = np.squeeze((split[1]) * target_daily_vol**2 / (wi.T @ np.diag(S) @ wi))

G = multf*G
cov_factor = (Z @ G @ Z.T) 
S = mults*S
cov_specific = np.diag(S)

cov = cov_factor + cov_specific 

beta = (np.diag(np.ones(N)) @ cov @ wi) / (wi.T @ cov @ wi)
# check the range of obtained betas
print(beta.min(), beta.max())

0.3678584946530952 1.6767607059793588


In [501]:
# risk
total_risk = np.sqrt(wi.T @ cov @ wi)*np.sqrt(252)
factor_risk = np.sqrt(wi.T @ (Z @ G @ Z.T) @ wi)*np.sqrt(252)
specific_risk = np.sqrt(wi.T @ np.diag(S) @ wi)*np.sqrt(252)
print(f"Total risk: {total_risk[0,0]:.4f}")
print(f"Factor risk: {factor_risk[0,0]:.4f}  ratio : {factor_risk[0,0]**2/total_risk[0,0]**2:.2%}")
print(f"Specific risk: {specific_risk[0,0]:.4f}  ratio : {specific_risk[0,0]**2/total_risk[0,0]**2:.2%}")   

Total risk: 0.1000
Factor risk: 0.0775  ratio : 60.00%
Specific risk: 0.0632  ratio : 40.00%


In [519]:
up_stocks = rng.choice(N, size=n_up, replace=False)
remaining = np.setdiff1d(np.arange(N), up_stocks)
down_stocks = rng.choice(remaining, size=n_down, replace=False)

trend = np.zeros(N)
trend[up_stocks] = rng.uniform(0.1, 1.0, size=n_up) * trend_daily
trend[down_stocks] = -rng.uniform(0.1, 1.0, size=n_down) * trend_daily


# generate the multivariate normal returns with the specified mean (trend) and covariance
rng = np.random.default_rng()
df_returns = pd.DataFrame(rng.multivariate_normal(trend, cov, size=T), index=dates, columns=cols)



In [503]:
# Running some diagnostics

realized_avg_vol = df_returns.mean(axis=1).std(ddof=1)
realized_avg_mean = df_returns.mean(axis=1).mean()

# sample_cov = np.cov(df_returns, rowvar=False, ddof=1)

theoretical_corr = cov / np.sqrt(np.outer(np.diag(cov), np.diag(cov)))
realized_corr = np.corrcoef(df_returns, rowvar=False)

mask_corr = ~np.eye(N, dtype=bool)
avg_theoretical_corr = theoretical_corr[mask_corr].mean()
avg_realized_corr = realized_corr[mask_corr].mean()

print("Target avg ann  vol:        ", target_daily_vol * np.sqrt(252))
print("Realized avg ann vol:       ", float(realized_avg_vol * np.sqrt(252)))
print("Realized avg ann mean:      ", float(realized_avg_mean * 252))

print("Realized avg ann mean up  :      ", float(df_returns.iloc[:,up_stocks].mean(axis=1).mean() * 252))
print("Realized avg ann mean down:      ", float(df_returns.iloc[:,down_stocks].mean(axis=1).mean() * 252))

print("Avg theoretical correlation: ", float(avg_theoretical_corr))
print("Avg realized correlation:    ", float(avg_realized_corr))

Target avg ann  vol:         0.1
Realized avg ann vol:        0.09894178685616074
Realized avg ann mean:       0.04681475235362271
Realized avg ann mean up  :       0.2711512263893701
Realized avg ann mean down:       -0.2938805172558817
Avg theoretical correlation:  0.004343688611999355
Avg realized correlation:     0.004233195845312916


In [504]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import importlib
import src.secondary_model_fast

importlib.reload(src.secondary_model_fast)

<module 'src.secondary_model_fast' from '/Users/ipacskornel/Downloads/Thesis/For github/meta_labeling_project/src/secondary_model_fast.py'>

In [505]:
from src.primary_strategy import build_primary_momentum_strategy_fast

pnl, wt, signal = build_primary_momentum_strategy_fast(
    df_returns=df_returns,
    lookback=260
)

In [506]:
from src.triple_barrier_fast import simple_triple_barrier_labels_fast

labels_rolling_30 = simple_triple_barrier_labels_fast(
    df_returns=df_returns,
    wt=wt,
    horizon=30,
    vol_window=20
)

In [507]:

from src.secondary_model_fast import (
    build_secondary_dataset_fast,
    purged_time_split,
    fit_logistic,
    fit_random_forest,
    fit_gradient_boosting,
    roc_auc_table,
    apply_probability_threshold,
    threshold_diagnostics,
    create_filtered_weights_fast,
    calculate_strategy_pnl
)

In [508]:
secondary_data = build_secondary_dataset_fast(
    df_returns = df_returns,
    labels = labels_rolling_30,
    window= 150
)

# Split now 50-25-25, with the triple barrier horizon being accounted for

train_df, val_df, test_df, val_start, test_start = purged_time_split(
    df=secondary_data,
    horizon=30, # Change when using different horizons for meta label targets
    train_frac=0.50,
    val_frac=0.25
)

# Same in principle as before

# Previous simple feature set:
# feature_cols = [
#     "weight",
#     "ret_150",
#     "vol_150"
# ]

# Previous side-adjusted feature set:
# feature_cols = [
#     "side",
#     "abs_weight",
#     "signed_ret_150",
#     "vol_150"
# ]

feature_cols = [
    "side",
    "abs_weight",
    "signed_ret_150",
    "vol_150",
    "corr_mkt_150"
]

model, train_pred_df, val_pred_df, test_pred_df = fit_logistic(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_cols=feature_cols
)


In [509]:
val_probability_summary = roc_auc_table(val_pred_df)
test_probability_summary = roc_auc_table(test_pred_df)

val_probability_summary

,Metric,Value
0,ROC AUC,0.572324


In [510]:
test_probability_summary

,Metric,Value
0,ROC AUC,0.566489


### Nonlinear Secondary-Model Benchmarks

The following models use the same train-validation-test split and feature columns as the logistic secondary model. They are included as a simple check of whether the modest ROC AUC is mainly due to model linearity or noisy labels/features.

In [511]:
rf_model, rf_train_pred_df, rf_val_pred_df, rf_test_pred_df = fit_random_forest(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_cols=feature_cols,
    random_state=0
)

gb_model, gb_train_pred_df, gb_val_pred_df, gb_test_pred_df = fit_gradient_boosting(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_cols=feature_cols,
    random_state=0
)

In [512]:
model_auc_comparison = pd.DataFrame({
    "Model": ["Logistic regression", "Random forest", "Gradient boosting"],
    "Validation ROC AUC": [
        roc_auc_table(val_pred_df).loc[0, "Value"],
        roc_auc_table(rf_val_pred_df).loc[0, "Value"],
        roc_auc_table(gb_val_pred_df).loc[0, "Value"],
    ],
    "Test ROC AUC": [
        roc_auc_table(test_pred_df).loc[0, "Value"],
        roc_auc_table(rf_test_pred_df).loc[0, "Value"],
        roc_auc_table(gb_test_pred_df).loc[0, "Value"],
    ],
})

model_auc_comparison

,Model,Validation ROC AUC,Test ROC AUC
0,Logistic regression,0.572324,0.566489
1,Random forest,0.570827,0.565320
2,Gradient boosting,0.572398,0.566650


In [513]:
# Enforcing indicator rule

threshold = 0.5 # Changeable

val_filtered_df = apply_probability_threshold(
    df=val_pred_df,
    threshold=threshold
)

# For later with threshold selection, only use validation results!!!
test_filtered_df = apply_probability_threshold(
    df=test_pred_df,
    threshold=threshold
)

In [514]:
# Filtering check for val and test

val_threshold_summary = threshold_diagnostics(val_filtered_df)
test_threshold_summary = threshold_diagnostics(test_filtered_df)

val_threshold_summary

,Metric,Value
0,Total trades,1.198500e+06
1,Trades kept,6.154750e+05
2,Trades removed,5.830250e+05
3,Fraction kept,5.135378e-01
4,Base positive rate,2.610972e-01
5,Positive rate among kept trades,3.033056e-01


In [515]:
# Keeping only test dates for evaluation

test_dates = pd.Index(test_filtered_df["t0"].unique())
test_dates = test_dates.intersection(wt.index)
test_dates = test_dates.intersection(df_returns.index)
test_dates = test_dates.sort_values()

test_filtered_df = test_filtered_df[
    test_filtered_df["t0"].isin(test_dates)
].copy()

In [516]:
from src.evaluation_metrics import evaluate_strategy

# Evaluate primary strategy on test dates using the relevant functions

primary_test_wt = wt.reindex(test_dates)

primary_test_pnl = calculate_strategy_pnl(
    weights=primary_test_wt,
    returns=df_returns
)

primary_test_summary = evaluate_strategy(
    returns_df=primary_test_pnl,
    weights_df=primary_test_wt
)

In [517]:
# Same evaluation for the added secondary filter

meta_test_wt = create_filtered_weights_fast(
    primary_weights=primary_test_wt,
    filtered_df=test_filtered_df
)

meta_test_pnl = calculate_strategy_pnl(
    weights=meta_test_wt,
    returns=df_returns
)

meta_test_summary = evaluate_strategy(
    returns_df=meta_test_pnl,
    weights_df=meta_test_wt
)

In [518]:
# Comparison of the two

strategy_comparison = primary_test_summary.merge(
    meta_test_summary,
    on="Metric",
    suffixes=("_primary", "_meta")
)

strategy_comparison

,Metric,Value_primary,Value_meta
0,Annualized return,0.012111,0.026704
1,Annualized volatility,0.078313,0.065630
2,Sharpe ratio,0.192872,0.434362
3,Maximum drawdown,-0.162595,-0.127544
4,Average daily return,0.000060,0.000113
5,Profit/loss ratio,0.981111,1.032600
6,Average daily turnover,0.043487,0.023140
7,Annualized turnover,10.958702,5.831195
